In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

from lightgbm import LGBMClassifier

### Load data

In [5]:
df_train = pd.read_csv('./data/train.csv', index_col='id')
df_test = pd.read_csv('./data/test.csv', index_col='id')
df_train.shape, df_test.shape

((11504798, 11), (7669866, 10))

### Minor changes to loaded data
- change column names to lowercase
- remove anomalous row from train data (unusual 'region_code' value)

In [6]:
df_train.rename(columns=lambda col : col.lower(), inplace=True)
df_test.rename(columns=lambda col : col.lower(), inplace=True)

drop_idx = df_train.loc[df_train['region_code'] == 39.2].index[0]
df_train.drop(index=drop_idx, inplace=True)
df_train.reset_index(drop=True, inplace=True)

df_train.head(10)

,gender,age,driving_license,region_code,previously_insured,vehicle_age,vehicle_damage,annual_premium,policy_sales_channel,vintage,response
id,,,,,,,,,,,
0,Male,21,1,35.0,0,1-2 Year,Yes,65101.0,124.0,187,0
1,Male,43,1,28.0,0,> 2 Years,Yes,58911.0,26.0,288,1
2,Female,25,1,14.0,1,< 1 Year,No,38043.0,152.0,254,0
3,Female,35,1,1.0,0,1-2 Year,Yes,2630.0,156.0,76,0
4,Female,36,1,15.0,1,1-2 Year,No,31951.0,152.0,294,0
5,Female,31,1,47.0,1,< 1 Year,No,28150.0,152.0,197,0
6,Male,23,1,45.0,1,< 1 Year,No,27128.0,152.0,190,0
7,Female,47,1,8.0,0,1-2 Year,Yes,40659.0,26.0,262,1
8,Female,26,1,28.0,1,< 1 Year,No,31639.0,152.0,36,0


### Data preprocessing
- make changes to annual_premium and vintage features
    - depending on their respective counts, those unique values that appear very few times (<50) will be replaced wherever they appear by the value -1
    - make weight features that are essentially counts for the row's corresponding feature value
- make new "composite" features
    - 2 new features are:
        - combine annual_premium with insurance
        - combine vehicle_age with insurance
    - these features are made by combining strings of the original features' values, and then ordinal encoding them

In [8]:
# change dtypes for each column to reduce data storage size
    # apply function to both train and test sets (involves no fitting or transforming so it is safe to simply run function twice)
    
def simple_preprocessing(df):
    df = df.copy()

    df['age'] = df['age'].astype('int8')
    df['driving_license'] = df['driving_license'].astype('int8')
    df['region_code'] = df['region_code'].astype('int8')
    df['previously_insured'] = df['previously_insured'].astype('int8')
    df['policy_sales_channel'] = df['policy_sales_channel'].astype('int16')
    df['gender'] = df['gender'].map({'Female': 0, 'Male': 1}).astype('int8')
    df['vehicle_age'] = df['vehicle_age'].map({'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}).astype('int8')
    df['vehicle_damage'] = df['vehicle_damage'].map({'No': 0, 'Yes': 1}).astype('int8')
    
    df['annual_premium'] = df['annual_premium'].astype('int32')    
    # for rows with "low-frequency" annual_premium values (i.e. that value appears only a few times in whole dataset), change the annual_premium values to -1
    annprem_valcount_dict = df['annual_premium'].value_counts().to_dict()
    annprem_as_count = df['annual_premium'].map(annprem_valcount_dict).astype('int32')
    df['annual_premium'] = df['annual_premium'].where(annprem_as_count >= 50, -1).astype('int32')
    # create new feature indicating each row's annual_premium value's frequency
    df['annual_premium_weight'] = annprem_as_count

    # make same changes for vintage as we did for annual_premium
    df['vintage'] = df['vintage'].astype('int16')
    vintage_valcount_dict = df['vintage'].value_counts().to_dict()
    vintage_as_count = df['vintage'].map(vintage_valcount_dict).astype('int32')
    df['vintage'] = df['vintage'].where(vintage_as_count >= 50, -1).astype('int32')
    df['vintage_weight'] = vintage_as_count
        
    # make new composite features - combine string values of annual_premium and previously_insured and ordinal-encode them
    df['annual_premium_and_insurance'] = df['previously_insured'].astype(str) + df['annual_premium'].astype(str)
    df['annual_premium_and_insurance'] = pd.factorize(df['annual_premium_and_insurance'])[0] + 1
    # second new composite feature (vehicle_age and previously_insured)
    df['vehicle_age_and_insurance'] = df['previously_insured'].astype(str) + df['vehicle_age'].astype(str)
    df['vehicle_age_and_insurance'] = pd.factorize(df['vehicle_age_and_insurance'])[0] + 1

    return df


df = pd.concat([df_train, df_test])
df = simple_preprocessing(df)

train = df.iloc[:df_train.shape[0], :]
train['response'] = train['response'].astype(int)
test = df.iloc[df_train.shape[0]:, :].drop(columns=['response'])

train.to_csv('./data/train_preprocessed.csv')
test.to_csv("./data/test_preprocessed.csv")

((11504797, 15), (7669866, 14))

### Undersampling

In [7]:
# split train data into X_train, X_val and X_test
X, X_test = train_test_split(train, test_size=100000, random_state=1, stratify=train['response'])
X_train, X_val = train_test_split(X, test_size=100000, random_state=1, stratify=X['response'])

y_train = X_train.pop('response')
y_val = X_val.pop('response')
y_test = X_test.pop('response')

def undersample(X, y, s):
    # major refers to y=0 datapoints, minor refers to y=1 datapoints
    X_major = X[y == 0].copy()
    X_major['response'] = 0
    # drop duplicate rows that were generated during feature engineering
    X_major = X_major.drop_duplicates()
    y_major = X_major.pop('response')
    X_minor = X[y == 1]

    # take sample of majority (same size as minority)
    sample_size = len(X_minor)
    X_sample, X_rem, y_sample, y_rem = train_test_split(X_major, y_major, train_size=sample_size, random_state=s, stratify=X_major['age'])
    X_minimal = pd.concat([X_sample, X_minor], axis=0)
    y_minimal = pd.concat([y_sample, y[y == 1]])

    return X_minimal, y_minimal, X_rem, y_rem

X_minimal, y_minimal, X_rem, y_rem = undersample(X_train, y_train, 1)
y_minimal = y_minimal.astype(int)

X_minimal.shape, y_minimal.shape

In [16]:
X_minimal.to_csv('./data/X_undersample.csv')
y_minimal.to_csv('./data/y_undersample.csv')

### Creating alternate dataset (minimal undersampling + additional datapoints)

In [9]:
# decided on addition of 4 million extra datapoints(taken from X_rem) to the minimal undersampled set

n=4
X_addition, _, y_addition, _ = train_test_split(X_rem, y_rem, train_size=n*1000000, random_state=1)
X_min_plus_four = pd.concat([X_minimal, X_addition], axis=0)
y_min_plus_four = pd.concat([y_minimal, y_addition], axis=0)

X_min_plus_four.shape, y_min_plus_four.shape

In [ ]:
X_min_plus_four.to_csv('./data/X_plus_four.csv')
y_min_plus_four.to_csv('./data/y_plus_four.csv')

### OOF(out-of-fold) predictions for base models
- baseline models chosen: logistic regression('lr'), light gradient boost model('lgbm')

In [6]:
X_large = pd.read_csv("./data/X_plus_four.csv", index_col=[0])
X_large.reset_index(drop=True, inplace=True)
X_large.head(10)

,gender,age,driving_license,region_code,previously_insured,vehicle_age,vehicle_damage,annual_premium,policy_sales_channel,vintage,annual_premium_weight,vintage_weight,annual_premium_and_insurance,vehicle_age_and_insurance
0,1,35,1,28,0,1,1,51660,124,283,773,64012,28992,1
1,0,25,1,41,1,0,0,28920,152,190,4944,61605,7925,3
2,0,56,1,28,0,1,1,52493,55,299,1211,44124,9327,1
3,1,48,1,5,1,1,0,25644,124,62,1374,38024,13654,4
4,1,29,1,30,0,0,1,2630,163,78,3519360,58730,4,5
5,0,21,1,46,1,0,0,30669,160,126,79,128514,30584,3
6,1,48,1,28,0,1,1,55409,26,171,1303,55047,6551,1
7,0,27,1,37,1,0,0,35661,152,58,3546,38586,14704,3
8,0,21,1,8,0,0,1,49807,160,93,587,42978,31577,5
9,1,63,1,39,0,1,1,2630,124,131,3519360,152981,4,1


In [7]:
y_large = pd.read_csv("./data/y_plus_four.csv", index_col=[0])
y_large.reset_index(drop=True, inplace=True)
y_large = y_large.values
y_large.head(10)

,response
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
5,0.0
6,0.0
7,0.0
8,0.0
9,0.0


In [34]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

def cross_validate_model(model, X, y, name, is_boost=False):
    oof_prob = np.zeros(shape=(X.shape[0]))
    for i, (train_idx, val_idx) in enumerate(skf.split(X,y)):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y[train_idx]
        y_val = y[val_idx]
        
        if is_boost:
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)],  eval_metric="auc")
        else:
            model.fit(X_train, y_train)
        preds = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, preds)
        print("fold {} AUC: {:.4f}".format(i, score))
        oof_prob[val_idx] = preds

    overall_score = roc_auc_score(y, oof_prob)
    print("overall AUC: {:.4f}".format(overall_score))
    
    save_data = pd.Series(data=oof_prob, index=X.index, name="response")
    file_path = f"./model_oof_preds/{name}_oof_preds.csv"
    save_data.to_csv(file_path)

In [30]:
# logistic regression baseline

num_cols = ['age', 'annual_premium', 'vintage', 'annual_premium_weight', 'vintage_weight']
cat_cols = ['vehicle_age', 'region_code', 'policy_sales_channel', 'annual_premium_and_insurance', 'vehicle_age_and_insurance']

step_1 = ColumnTransformer([('scaler', StandardScaler(), num_cols),
                            ('target_encoder', TargetEncoder(), cat_cols)],
                            remainder='passthrough').set_output(transform='pandas')

step_2 = ColumnTransformer([('poly', PolynomialFeatures(include_bias=False),
                            ['scaler__age', 'scaler__annual_premium', 'scaler__vintage'])],
                            remainder='passthrough').set_output(transform='pandas')

feature_transforms = Pipeline([('scaling_encoding', step_1),
                               ('poly_features', step_2)])

model_lr = Pipeline([('transformations', feature_transforms),
                       ('logistic_regression', LogisticRegression(C=0.1, solver='newton-cholesky'))])

cross_validate_model(model_lr, X_large, y_large, name="lr")

In [9]:
# LGBM baseline

lgbm_params = {'n_estimators': 1000,
               'max_depth': 6,
               'verbose':-1,
               "metric": "auc",
               "early_stopping_round": 50,
               #"max_bin": 262143,
               'num_leaves': 223,
               'learning_rate': 0.028095688623590447,
               'min_child_samples': 54,
               'subsample': 0.5395472919165504,
               'colsample_bytree': 0.547518064129546,
               'lambda_l1': 3.4444245446562,
               'lambda_l2': 2.87490408088595e-05}

lgbm_model = LGBMClassifier(**lgbm_params)

cross_validate_model(lgbm_model, X_large, y_large.values.flatten(), name="lgbm", is_boost=True)